In [1]:
# import numpy as np
# from dm_control import suite
# from dm_control import viewer
# from neurobo.ncap.envs import swim  # Adjust the import path as necessary

# # Create the swimmer environment
# env = swim()

# # Print action specification details
# action_spec = env.action_spec()
# print(f"Action Spec: Minimum: {action_spec.minimum}, Maximum: {action_spec.maximum}, Shape: {action_spec.shape}")

# # Initialize the agent (this is a simple random agent for demonstration purposes)
# class RandomAgent:
#     def __init__(self, action_spec):
#         self.action_spec = action_spec

#     def select_action(self):
#         action = np.random.uniform(self.action_spec.minimum, self.action_spec.maximum, self.action_spec.shape)
#         print(f"Action: {action}, Type: {type(action)}, Dtype: {action.dtype}, Shape: {action.shape}")
#         return action.astype(np.float64)

# # Create an instance of the agent
# agent = RandomAgent(env.action_spec())

# # Define a custom loop to step the environment and update the viewer
# def custom_loop(env, agent):
#     time_step = env.reset()
#     while not time_step.last():
#         action = agent.select_action()
#         time_step = env.step(action)
#         print(f"Reward: {time_step.reward}, Observation: {time_step.observation}")

# # Launch the viewer with the custom loop
# viewer.launch(env, policy=lambda ts: custom_loop(env, agent))

In [2]:
import numpy as np
import collections
from gymnasium.envs.registration import register
import gymnasium as gym
import dm_control.suite.swimmer as swimmer
from dm_control.rl import control
from dm_control.utils import rewards

# Horizontal speed above which the reward is 1.
_SWIM_SPEED = 0.1

@swimmer.SUITE.add()
def free_swim_reece(
  n_links=6,
  desired_speed=_SWIM_SPEED,
  time_limit=swimmer._DEFAULT_TIME_LIMIT,
  random=None,
  environment_kwargs={},
):
  """Returns the Swim task for a n-link swimmer."""
  model_string, assets = swimmer.get_model_and_assets(n_links)

  physics = swimmer.Physics.from_xml_string(model_string, assets=assets)
  task = Swim(desired_speed=desired_speed, random=random)
  return control.Environment(
    physics,
    task,
    time_limit=time_limit,
    control_timestep=swimmer._CONTROL_TIMESTEP,
    **environment_kwargs,
  )

class Swim(swimmer.Swimmer):
  """Task to swim forwards at the desired speed."""
  def __init__(self, desired_speed=_SWIM_SPEED, **kwargs):
    super().__init__(**kwargs)
    self._desired_speed = desired_speed
    self._previous_position = None
    self._total_distance_traveled = 0.0

  def initialize_episode(self, physics):
    super().initialize_episode(physics)
    # Hide target by setting alpha to 0.
    physics.named.model.mat_rgba['target', 'a'] = 0
    physics.named.model.mat_rgba['target_default', 'a'] = 0
    physics.named.model.mat_rgba['target_highlight', 'a'] = 0
    self._previous_position = physics.data.qpos.copy()

  def get_observation(self, physics):
    """Returns an observation of joint angles and body velocities."""
    obs = collections.OrderedDict()
    obs['joints'] = physics.joints()
    obs['body_velocities'] = physics.body_velocities()
    return obs

  def get_reward(self, physics):
    """Returns a smooth reward that is 0 when stopped or moving backwards, and rises linearly to 1
    when moving forwards at the desired speed."""
    forward_velocity = -physics.named.data.sensordata['head_vel'][1]
    return rewards.tolerance(
      forward_velocity,
      bounds=(self._desired_speed, float('inf')),
      margin=self._desired_speed,
      value_at_margin=0.,
      sigmoid='linear',
    )

  # def get_reward(self, physics):
  #   """Returns a reward proportional to the total distance traveled."""
  #   current_position = physics.data.qpos.copy()
  #   distance_traveled = np.linalg.norm(current_position - self._previous_position)
  #   self._total_distance_traveled += distance_traveled
  #   self._previous_position = current_position
  #   return distance_traveled
  
# Register the custom environment

# Now test the registration and environment creation


In [3]:
import gym
from gym import spaces
import numpy as np
import dm_control.suite as suite
from dm_control.rl import control
import collections
import dm_control.suite.swimmer as swimmer
import time

class DmControlToGymWrapper(gym.Env):
    def __init__(self, dm_env):
        super(DmControlToGymWrapper, self).__init__()
        self.dm_env = dm_env

        # Define action and observation space
        self.action_space = spaces.Box(low=-1.0, high=1.0, shape=(self.dm_env.action_spec().shape[0],), dtype=np.float32)
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(self.dm_env.observation_spec()['joints'].shape[0] +
                                                                            self.dm_env.observation_spec()['body_velocities'].shape[0],), dtype=np.float32)

    def reset(self):
        # Reset the environment and return the initial observation
        time_step = self.dm_env.reset()
        return self._get_observation(time_step)

    def step(self, action):
        # Step the environment with the given action
        time_step = self.dm_env.step(action)
        observation = self._get_observation(time_step)
        reward = self.dm_env.task.get_reward(self.dm_env.physics)
        done = time_step.last()
        return observation, reward, done, {}

    def _get_observation(self, time_step):
        """Converts the time_step to a Gym-compatible observation."""
        obs = collections.OrderedDict()
        obs['joints'] = time_step.observation['joints']
        obs['body_velocities'] = time_step.observation['body_velocities']
        return np.concatenate([obs['joints'], obs['body_velocities']], axis=-1)
    
task = swimmer.SUITE['free_swim_reece']
env = task()
gym_env = DmControlToGymWrapper(env)

# Optionally, wrap it in a vectorized environment for Stable-Baselines3
#from stable_baselines3.common.vec_env import DummyVecEnv
#gym_env = DummyVecEnv([lambda: gym_env])

# Train an agent using PPO from Stable-Baselines3
import stable_baselines3 as sb3

model = sb3.PPO("MlpPolicy", gym_env, verbose=1)
model.learn(total_timesteps=10000)

# Save the trained model
model.save("ppo_swimmer_model")

# Optionally, evaluate the agent
obs = gym_env.reset()
for _ in range(1000):
    action, _ = model.predict(obs)
    obs, reward, done, info = gym_env.step(action)
    
    if done:
        obs = gym_env.reset()

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


/Users/reecekeller/miniforge3/envs/mujoco_env/lib/python3.10/site-packages/stable_baselines3/common/vec_env/patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 1e+03    |
|    ep_rew_mean     | 9.48e+04 |
| time/              |          |
|    fps             | 5158     |
|    iterations      | 1        |
|    time_elapsed    | 0        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 1e+03       |
|    ep_rew_mean          | 1.86e+05    |
| time/                   |             |
|    fps                  | 3454        |
|    iterations           | 2           |
|    time_elapsed         | 1           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.004779415 |
|    clip_fraction        | 0.0179      |
|    clip_range           | 0.2         |
|    entropy_loss         | -7.1        |
|    explained_variance   | -3.46e-05   |
|    learning_rate        | 0.

In [4]:
# Load the trained model
from dm_control import viewer
model = sb3.PPO.load("ppo_swimmer_model")

# Define a function to run the policy and visualize using viewer.launch
def visualize_policy(env, model, num_steps=1000):
    def policy_step(time_step):
        obs = np.concatenate([time_step.observation['joints'], time_step.observation['body_velocities']], axis=-1)
        action, _ = model.predict(obs)
        return action

    viewer.launch(env, policy=policy_step)

# Visualize the policy
visualize_policy(env, model)

2024-11-20 15:14:21.175 python[39655:2712256] +[IMKClient subclass]: chose IMKClient_Modern
2024-11-20 15:14:21.175 python[39655:2712256] +[IMKInputSession subclass]: chose IMKInputSession_Modern


KeyboardInterrupt: 

In [ ]:
print(swimmer.SUITE)

TaggedTasks(OrderedDict([('swimmer6', <function swimmer6 at 0x127b4f1c0>), ('swimmer15', <function swimmer15 at 0x127b4f250>), ('free_swim_reece', <function free_swim_reece at 0x1301695a0>)]))


In [ ]:
import gym
print(gym.envs.registry.values())  # This will print all the registered environments

dict_values([EnvSpec(id='CartPole-v0', entry_point='gym.envs.classic_control.cartpole:CartPoleEnv', reward_threshold=195.0, nondeterministic=False, max_episode_steps=200, order_enforce=True, autoreset=False, disable_env_checker=False, apply_api_compatibility=False, kwargs={}, namespace=None, name='CartPole', version=0), EnvSpec(id='CartPole-v1', entry_point='gym.envs.classic_control.cartpole:CartPoleEnv', reward_threshold=475.0, nondeterministic=False, max_episode_steps=500, order_enforce=True, autoreset=False, disable_env_checker=False, apply_api_compatibility=False, kwargs={}, namespace=None, name='CartPole', version=1), EnvSpec(id='MountainCar-v0', entry_point='gym.envs.classic_control.mountain_car:MountainCarEnv', reward_threshold=-110.0, nondeterministic=False, max_episode_steps=200, order_enforce=True, autoreset=False, disable_env_checker=False, apply_api_compatibility=False, kwargs={}, namespace=None, name='MountainCar', version=0), EnvSpec(id='MountainCarContinuous-v0', entry_p

In [ ]:
import gym

# Now you can use the custom environment ID to create an instance of your environment
env = gym.make('CustomSwimmer-v0')

# Optionally, you can use it with Stable-Baselines3 for training
import stable_baselines3 as sb3
model = sb3.PPO("MlpPolicy", env, verbose=1)
model.learn(total_timesteps=10000)

NameNotFound: Environment CustomSwimmer doesn't exist. Did you mean: `Swimmer`?